# PatchCore ViT-B/16 One-Layer Defect-Tuning (`x224`)

This notebook is the canonical training and evaluation workflow for the one-layer ViT-B/16 PatchCore experiment with defect-aware threshold tuning.

Run this notebook from top to bottom to reproduce the experiment locally. The notebook prepares the data split, trains the model when needed, evaluates the trained model, and writes outputs into the experiment-local artifact folders.

Artifacts written by this notebook:
- `[experiments/anomaly_detection/patchcore/vit_b16/x224/one_layer_defect_tuning/artifacts/checkpoints](experiments/anomaly_detection/patchcore/vit_b16/x224/one_layer_defect_tuning/artifacts/checkpoints)` for model checkpoints
- `[experiments/anomaly_detection/patchcore/vit_b16/x224/one_layer_defect_tuning/artifacts/plots](experiments/anomaly_detection/patchcore/vit_b16/x224/one_layer_defect_tuning/artifacts/plots)` for saved figures
- `[experiments/anomaly_detection/patchcore/vit_b16/x224/one_layer_defect_tuning/artifacts/results](experiments/anomaly_detection/patchcore/vit_b16/x224/one_layer_defect_tuning/artifacts/results)` for metrics, score files, and CSV outputs


## Setup

This cell installs or checks optional notebook dependencies before the rest of the workflow runs.


In [ ]:
import importlib
import importlib.util
import subprocess
import sys
for pkg in ['timm', 'tqdm']:
    if importlib.util.find_spec(pkg) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])
print('timm + tqdm ready')


## Imports

This cell imports the Python packages used across training, evaluation, plotting, and artifact export.


In [ ]:
import os, gc, json, random, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import timm
from tqdm.auto import tqdm
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_CUDA = DEVICE.type == 'cuda'
if USE_CUDA:
    torch.backends.cudnn.benchmark = True
    torch.set_float32_matmul_precision('high')
print('Device:', DEVICE)
if USE_CUDA:
    p = torch.cuda.get_device_properties(0)
    print(f'GPU: {p.name}  VRAM: {p.total_memory / 1000000000.0:.1f} GB')
from IPython.display import display


## Configuration

This cell resolves the repo root, defines experiment settings, and points all outputs to the local artifact folders.


In [ ]:
from pathlib import Path
import sys
cwd = Path.cwd().resolve()
candidate_roots = [cwd, *cwd.parents]
PROJECT_ROOT = None
for candidate in candidate_roots:
    if (candidate / 'src' / 'wafer_defect').exists() and (candidate / 'configs').exists():
        PROJECT_ROOT = candidate
        break
if PROJECT_ROOT is None:
    raise RuntimeError('Could not locate repo root containing src/wafer_defect and configs/')
SRC_ROOT = PROJECT_ROOT / 'src'
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))
DATA_PATH = str(PROJECT_ROOT / 'data' / 'raw' / 'LSWMD.pkl')
IMAGE_SIZE = 224
TRAIN_NORMAL_N = 40000
TUNE_NORMAL_N = 5000
TUNE_DEFECT_N = 5000
TEST_NORMAL_N = 5000
TEST_DEFECT_N = 250
VIT_FEATURE_BLOCK = 6
PATCH_EMBED_DIM = 256
MEMORY_BANK_MAX_PATCHES = 600000
SCORE_CHUNK = 512
PATCHCORE_NN_K = 3
TOPK_PATCH_RATIO = 0.03
BATCH_SIZE = 128
NUM_WORKERS = 0
THRESHOLD_PERCENTILE_MIN = 90
THRESHOLD_PERCENTILE_MAX = 99.9
THRESHOLD_PERCENTILE_STEPS = 100
THRESHOLD_GRID_STEPS = 300
FORCE_REBUILD_SCORES = False
FORCE_RERUN_UMAP = False
ARTIFACT_DIR = str(PROJECT_ROOT / 'experiments/anomaly_detection/patchcore/vit_b16/x224/one_layer_defect_tuning/artifacts')
CHECKPOINTS_DIR = os.path.join(ARTIFACT_DIR, 'checkpoints')
PLOTS_DIR = os.path.join(ARTIFACT_DIR, 'plots')
RESULTS_DIR = os.path.join(ARTIFACT_DIR, 'results')
MODEL_EXPORT_PATH = os.path.join(CHECKPOINTS_DIR, 'model.pt')
METRICS_EXPORT_PATH = os.path.join(RESULTS_DIR, 'evaluation_metrics.json')
SUMMARY_EXPORT_PATH = os.path.join(RESULTS_DIR, 'summary.json')
SCORES_EXPORT_PATH = os.path.join(RESULTS_DIR, 'scores.npz')
UMAP_PNG_PATH = os.path.join(PLOTS_DIR, 'umap_test_embeddings.png')
UMAP_CSV_PATH = os.path.join(RESULTS_DIR, 'umap_test_embeddings.csv')
SCORE_DISTRIBUTION_PNG_PATH = os.path.join(PLOTS_DIR, 'score_distribution_test.png')
CONFUSION_MATRIX_PNG_PATH = os.path.join(PLOTS_DIR, 'confusion_matrix_test.png')
THRESHOLD_SELECTION_PNG_PATH = os.path.join(PLOTS_DIR, 'threshold_selection_tune.png')
CLASSIFICATION_REPORT_PATH = os.path.join(RESULTS_DIR, 'classification_report.txt')
DEFECT_RECALL_CSV_PATH = os.path.join(RESULTS_DIR, 'test_defect_recall.csv')
TEST_SCORES_CSV_PATH = os.path.join(RESULTS_DIR, 'test_scores.csv')
for path_str in [CHECKPOINTS_DIR, PLOTS_DIR, RESULTS_DIR]:
    os.makedirs(path_str, exist_ok=True)
LEGACY_MODEL_EXPORT_PATH = os.path.join(ARTIFACT_DIR, 'model.pt')
if os.path.exists(MODEL_EXPORT_PATH) or os.path.exists(LEGACY_MODEL_EXPORT_PATH):
    checkpoint_probe_path = MODEL_EXPORT_PATH if os.path.exists(MODEL_EXPORT_PATH) else LEGACY_MODEL_EXPORT_PATH
    try:
        checkpoint_probe = torch.load(checkpoint_probe_path, map_location='cpu')
        ckpt_cfg = checkpoint_probe.get('config', {}) if isinstance(checkpoint_probe, dict) else {}
        if 'patch_embed_dim' in ckpt_cfg:
            PATCH_EMBED_DIM = int(ckpt_cfg['patch_embed_dim'])
        if 'vit_feature_block' in ckpt_cfg:
            VIT_FEATURE_BLOCK = int(ckpt_cfg['vit_feature_block'])
        print(f'Aligned notebook config to saved checkpoint: block={VIT_FEATURE_BLOCK}, embed_dim={PATCH_EMBED_DIM}')
    except Exception as exc:
        print(f'Warning: could not probe saved checkpoint config: {exc}')
print('Artifacts will be saved to:', ARTIFACT_DIR)
print(f'ViT block={VIT_FEATURE_BLOCK}  embed_dim={PATCH_EMBED_DIM}')
print(f'Bank cap={MEMORY_BANK_MAX_PATCHES:,}  batch={BATCH_SIZE}  workers={NUM_WORKERS}')
print(f'Test defect ratio: {TEST_DEFECT_N / TEST_NORMAL_N * 100:.1f}%  ({TEST_DEFECT_N} / {TEST_NORMAL_N})')
print(f'FORCE_REBUILD_SCORES={FORCE_REBUILD_SCORES}  FORCE_RERUN_UMAP={FORCE_RERUN_UMAP}')


## Load Dataset

This cell loads the legacy WM-811K pickle, cleans labels, and prepares the dataframe used for the experiment split.


In [ ]:
from wafer_defect.data.legacy_pickle import read_legacy_pickle
df = read_legacy_pickle(DATA_PATH)
print('Raw shape:', df.shape)

def parse_failure_label(v):
    if v is None:
        return 'unknown'
    if isinstance(v, float) and np.isnan(v):
        return 'unknown'
    if isinstance(v, (list, tuple, np.ndarray)):
        a = np.array(v).reshape(-1)
        return 'unknown' if len(a) == 0 else str(a[0])
    return str(v)
df = df.copy()
df['failure_label'] = df['failureType'].apply(parse_failure_label).str.strip()
invalid = {'0', 'unknown', 'nan', 'None', '[]'}
df = df[~df['failure_label'].isin(invalid)].copy()
df['is_anomaly'] = (df['failure_label'].str.lower() != 'none').astype(int)
normal_df = df[df['is_anomaly'] == 0].copy()
defect_df = df[df['is_anomaly'] == 1].copy()
print(f'Labeled: {len(df):,}   Normal: {len(normal_df):,}   Defect: {len(defect_df):,}')
print('\nDefect breakdown:')
print(defect_df['failure_label'].value_counts())


## -- 4. Split ------------------------------------------------------------------

This cell continues the experiment workflow and writes its outputs into the local artifact folders.


In [ ]:
req_n = TRAIN_NORMAL_N + TUNE_NORMAL_N + TEST_NORMAL_N
req_d = TUNE_DEFECT_N + TEST_DEFECT_N
if len(normal_df) < req_n:
    raise ValueError(f'Need {req_n:,} normals, have {len(normal_df):,}')
if len(defect_df) < req_d:
    raise ValueError(f'Need {req_d:,} defects, have {len(defect_df):,}')
rng = np.random.default_rng(SEED)
ns = normal_df.iloc[rng.permutation(len(normal_df))].reset_index(drop=True)
ds = defect_df.iloc[rng.permutation(len(defect_df))].reset_index(drop=True)
a = TRAIN_NORMAL_N
b = a + TUNE_NORMAL_N
c = b + TEST_NORMAL_N
train_normal_df = ns.iloc[0:a].copy()
tune_normal_df = ns.iloc[a:b].copy()
test_normal_df = ns.iloc[b:c].copy()
tune_defect_df = ds.iloc[0:TUNE_DEFECT_N].copy()
test_defect_df = ds.iloc[TUNE_DEFECT_N:TUNE_DEFECT_N + TEST_DEFECT_N].copy()
del df, normal_df, defect_df, ns, ds
gc.collect()
print(f'Train normal : {len(train_normal_df):>7,}')
print(f'Tune  normal : {len(tune_normal_df):>7,}')
print(f'Tune  defect : {len(tune_defect_df):>7,}')
print(f'Test  normal : {len(test_normal_df):>7,}')
print(f'Test  defect : {len(test_defect_df):>7,}')


## -- 5. Lazy WaferDataset ------------------------------------------------------

This cell continues the experiment workflow and writes its outputs into the local artifact folders.


In [ ]:
class WaferDataset(Dataset):

    def __init__(self, frame: pd.DataFrame, size: int=224):
        self.maps = frame['waferMap'].values
        self.labels = frame['is_anomaly'].values.astype(np.int64)
        self.size = size

    def __len__(self):
        return len(self.maps)

    def __getitem__(self, idx):
        arr = np.clip(np.array(self.maps[idx], dtype=np.int64), 0, 2)
        x = torch.tensor(arr, dtype=torch.long)
        x = F.one_hot(x, num_classes=3).permute(2, 0, 1).float()
        x = F.interpolate(x.unsqueeze(0), size=(self.size, self.size), mode='nearest').squeeze(0)
        return (x, int(self.labels[idx]))
loader_kw = dict(batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=USE_CUDA, persistent_workers=NUM_WORKERS > 0)
train_loader = DataLoader(WaferDataset(train_normal_df, IMAGE_SIZE), **loader_kw)
tune_normal_loader = DataLoader(WaferDataset(tune_normal_df, IMAGE_SIZE), **loader_kw)
tune_defect_loader = DataLoader(WaferDataset(tune_defect_df, IMAGE_SIZE), **loader_kw)
test_normal_loader = DataLoader(WaferDataset(test_normal_df, IMAGE_SIZE), **loader_kw)
test_defect_loader = DataLoader(WaferDataset(test_defect_df, IMAGE_SIZE), **loader_kw)
for name, ldr in [('train_normal', train_loader), ('tune_normal', tune_normal_loader), ('tune_defect', tune_defect_loader), ('test_normal', test_normal_loader), ('test_defect', test_defect_loader)]:
    print(f'{name:<14}: {len(ldr):>4} batches')
xb, yb = next(iter(train_loader))
print(f'\nBatch shape: {tuple(xb.shape)}  dtype={xb.dtype}  sample labels={yb[:4].tolist()}')


## Define Model

This cell defines the PatchCore feature extractor and supporting model components for this branch.


In [ ]:
class ViTPatchExtractor(nn.Module):

    def __init__(self, block_idx: int=VIT_FEATURE_BLOCK, proj_dim: int=PATCH_EMBED_DIM):
        super().__init__()
        self.vit = timm.create_model('vit_base_patch16_224.augreg_in21k_ft_in1k', pretrained=True, num_classes=0)
        self._feat = None
        self.vit.blocks[block_idx].register_forward_hook(lambda m, i, o: setattr(self, '_feat', o))
        self.proj = nn.Linear(768, proj_dim, bias=False)

    def forward(self, x):
        self.vit(x)
        return self._feat[:, 1:, :]
extractor = ViTPatchExtractor().to(DEVICE).eval()
for p in extractor.parameters():
    p.requires_grad = False
with torch.inference_mode():
    dummy = torch.zeros(2, 3, IMAGE_SIZE, IMAGE_SIZE, device=DEVICE)
    out = extractor(dummy)
    proj = extractor.proj(out)
print(f'ViT block-{VIT_FEATURE_BLOCK} output : {tuple(out.shape)}')
print(f'After projection         : {tuple(proj.shape)}')


## Train and Score

This cell builds the PatchCore memory bank, computes anomaly scores, and saves the score bundle needed for later evaluation.


In [ ]:
REQUIRED_SCORE_KEYS = {'train_scores_z', 'tune_normal_scores_z', 'tune_defect_scores_z', 'test_normal_scores_z', 'test_defect_scores_z', 'train_mu', 'train_std'}
PARTIAL_SCORE_KEYS = {'train_scores_z', 'tune_normal_scores_z', 'test_normal_scores_z', 'test_defect_scores_z', 'train_mu', 'train_std'}
HAS_SAVED_CHECKPOINT = os.path.exists(MODEL_LOAD_PATH)
checkpoint_artifact = None
if HAS_SAVED_CHECKPOINT:
    checkpoint_artifact = torch.load(MODEL_LOAD_PATH, map_location='cpu')
    extractor.load_state_dict(checkpoint_artifact['extractor_state_dict'])
    extractor = extractor.to(DEVICE).eval()
    print(f'Loaded saved checkpoint: {MODEL_LOAD_PATH}')
saved_score_keys = set()
if os.path.exists(SCORES_EXPORT_PATH):
    with np.load(SCORES_EXPORT_PATH) as existing_scores:
        saved_score_keys = set(existing_scores.files)
HAS_FULL_SAVED_SCORES = REQUIRED_SCORE_KEYS.issubset(saved_score_keys)
HAS_PARTIAL_SAVED_SCORES = PARTIAL_SCORE_KEYS.issubset(saved_score_keys)
REUSE_SAVED_RESULTS = HAS_SAVED_CHECKPOINT and HAS_FULL_SAVED_SCORES and (not FORCE_REBUILD_SCORES)
REUSE_PARTIAL_SAVED_RESULTS = HAS_SAVED_CHECKPOINT and HAS_PARTIAL_SAVED_SCORES and (not FORCE_REBUILD_SCORES)
memory_bank = None
memory_bank_t = None
if REUSE_SAVED_RESULTS or REUSE_PARTIAL_SAVED_RESULTS:
    print('Skipping memory bank rebuild and reusing saved local artifacts.')
else:

    def extract_embeddings(xb: torch.Tensor) -> torch.Tensor:
        """L2-normalised embeddings: [B*196, proj_dim] on GPU."""
        with torch.inference_mode():
            with torch.autocast('cuda', torch.float16, enabled=USE_CUDA):
                feat = extractor(xb)
                emb = extractor.proj(feat)
            emb = emb.float().reshape(-1, emb.shape[-1])
            emb = F.normalize(emb, p=2, dim=1)
        return emb
    sampled, total_seen, sample_ratio = ([], 0, None)
    print('Building memory bank..')
    bank_iter = tqdm(enumerate(train_loader), total=len(train_loader), desc='Bank build', unit='batch')
    for step, (xb, _) in bank_iter:
        xb = xb.to(DEVICE)
        emb = extract_embeddings(xb)
        total_seen += len(emb)
        if sample_ratio is None:
            tokens_per_img = len(emb) // len(xb)
            estimated_total = tokens_per_img * len(train_normal_df)
            sample_ratio = min(1.0, MEMORY_BANK_MAX_PATCHES / estimated_total)
            print(f'  Tokens/image : {tokens_per_img}')
            print(f'  Est. total   : {estimated_total:,}')
            print(f'  Sample ratio : {sample_ratio:.5f}')
        if sample_ratio < 1.0:
            k = max(1, int(round(len(emb) * sample_ratio)))
            idx = torch.randperm(len(emb), device=DEVICE)[:k]
            emb = emb[idx]
        sampled.append(emb)
        if (step + 1) % 20 == 0 or step + 1 == len(train_loader):
            n = sum((len(e) for e in sampled))
            bank_iter.set_postfix(bank_tokens=f'{n:,}')
    memory_bank = torch.cat(sampled, dim=0)
    del sampled
    gc.collect()
    if len(memory_bank) > MEMORY_BANK_MAX_PATCHES:
        idx = torch.randperm(len(memory_bank), device=DEVICE)[:MEMORY_BANK_MAX_PATCHES]
        memory_bank = memory_bank[idx]
    memory_bank = F.normalize(memory_bank, p=2, dim=1).contiguous()
    memory_bank_t = memory_bank.t().contiguous()
    mb_mb = memory_bank.element_size() * memory_bank.numel() / 1000000.0
    print(f'Final bank : {len(memory_bank):,} x {memory_bank.shape[1]}-d  ({mb_mb:.1f} MB VRAM)')


## Reload Scores

This cell reloads the saved score bundle so threshold selection and downstream evaluation use the persisted results.


In [ ]:
def min_dist_to_bank(patches, bank_t, chunk=512, k=3):
    out = []
    for i in range(0, len(patches), chunk):
        p = patches[i:i + chunk]
        sim = p @ bank_t
        kk = min(k, sim.shape[1])
        best = sim.topk(kk, dim=1).values
        dist = torch.sqrt(torch.clamp(2.0 - 2.0 * best, min=0.0)).mean(dim=1)
        out.append(dist)
    return torch.cat(out)

def score_loader(loader, bank_t, topk_ratio=0.1, nn_k=3, desc='Scoring'):
    scores = []
    with torch.inference_mode():
        pbar = tqdm(loader, total=len(loader), desc=desc, unit='batch')
        for xb, _ in pbar:
            xb = xb.to(DEVICE)
            with torch.autocast('cuda', torch.float16, enabled=USE_CUDA):
                feat = extractor(xb)
                emb = extractor.proj(feat)
            emb = emb.float().reshape(-1, emb.shape[-1])
            emb = F.normalize(emb, p=2, dim=1)
            ps = min_dist_to_bank(emb, bank_t, SCORE_CHUNK, nn_k)
            b = len(xb)
            ps = ps.reshape(b, -1)
            topk = max(1, int(round(ps.shape[1] * topk_ratio)))
            topk = min(topk, ps.shape[1])
            s = ps.topk(topk, dim=1).values.mean(dim=1)
            scores.append(s.cpu())
    return torch.cat(scores).numpy()
kw = dict(topk_ratio=TOPK_PATCH_RATIO, nn_k=PATCHCORE_NN_K)
if REUSE_SAVED_RESULTS or REUSE_PARTIAL_SAVED_RESULTS:
    with np.load(SCORES_EXPORT_PATH) as d:
        train_scores_z = d['train_scores_z']
        tune_normal_scores_z = d['tune_normal_scores_z']
        tune_defect_scores_z = d['tune_defect_scores_z'] if 'tune_defect_scores_z' in d.files else None
        test_normal_scores_z = d['test_normal_scores_z']
        test_defect_scores_z = d['test_defect_scores_z']
        mu = float(d['train_mu'])
        std = float(d['train_std'])
    print(f'Loaded saved score artifacts from: {SCORES_EXPORT_PATH}')
else:
    train_scores = score_loader(train_loader, memory_bank_t, desc='Score train-normal', **kw)
    tune_normal_scores = score_loader(tune_normal_loader, memory_bank_t, desc='Score tune-normal', **kw)
    tune_defect_scores = score_loader(tune_defect_loader, memory_bank_t, desc='Score tune-defect', **kw)
    test_normal_scores = score_loader(test_normal_loader, memory_bank_t, desc='Score test-normal', **kw)
    test_defect_scores = score_loader(test_defect_loader, memory_bank_t, desc='Score test-defect', **kw)
    mu = float(np.mean(train_scores))
    std = float(np.std(train_scores) + 1e-08)

    def znorm(x):
        return (x - mu) / std
    train_scores_z = znorm(train_scores)
    tune_normal_scores_z = znorm(tune_normal_scores)
    tune_defect_scores_z = znorm(tune_defect_scores)
    test_normal_scores_z = znorm(test_normal_scores)
    test_defect_scores_z = znorm(test_defect_scores)
    np.savez_compressed(SCORES_EXPORT_PATH, train_scores_z=train_scores_z, tune_normal_scores_z=tune_normal_scores_z, tune_defect_scores_z=tune_defect_scores_z, test_normal_scores_z=test_normal_scores_z, test_defect_scores_z=test_defect_scores_z, train_mu=np.array(mu), train_std=np.array(std))
    print(f'Scores saved. mu={mu:.6f}  sigma={std:.6f}')


## Reload Scores

This cell reloads the saved score bundle so threshold selection and downstream evaluation use the persisted results.


In [ ]:
with np.load(SCORES_EXPORT_PATH) as score_bundle:
    tune_normal_scores_z = score_bundle['tune_normal_scores_z']
    test_normal_scores_z = score_bundle['test_normal_scores_z']
    test_defect_scores_z = score_bundle['test_defect_scores_z']
    tune_defect_scores_z = score_bundle['tune_defect_scores_z'] if 'tune_defect_scores_z' in score_bundle.files else None
    mu = float(score_bundle['train_mu'])
    std = float(score_bundle['train_std'])
if tune_defect_scores_z is not None:
    y_tune = np.concatenate([np.zeros(len(tune_normal_scores_z), dtype=int), np.ones(len(tune_defect_scores_z), dtype=int)])
    score_tune = np.concatenate([tune_normal_scores_z, tune_defect_scores_z])
    grid_lo = float(np.percentile(score_tune, 1.0))
    grid_hi = float(np.percentile(score_tune, 99.0))
    threshold_grid = np.linspace(grid_lo, grid_hi, int(globals().get('THRESHOLD_GRID_STEPS', 300)))
    best_bal_acc = -1.0
    best_f1 = -1.0
    threshold_z = float(np.median(score_tune))
    for threshold_candidate in threshold_grid:
        pred = (score_tune > threshold_candidate).astype(int)
        tp = int(((pred == 1) & (y_tune == 1)).sum())
        tn = int(((pred == 0) & (y_tune == 0)).sum())
        fp = int(((pred == 1) & (y_tune == 0)).sum())
        fn = int(((pred == 0) & (y_tune == 1)).sum())
        tpr = tp / max(tp + fn, 1)
        tnr = tn / max(tn + fp, 1)
        bal_acc = 0.5 * (tpr + tnr)
        precision = tp / max(tp + fp, 1)
        recall = tpr
        f1 = 0.0 if precision + recall == 0 else 2.0 * precision * recall / (precision + recall)
        if bal_acc > best_bal_acc or (np.isclose(bal_acc, best_bal_acc) and f1 > best_f1):
            best_bal_acc = bal_acc
            best_f1 = f1
            threshold_z = float(threshold_candidate)
else:
    threshold_z = float(np.quantile(tune_normal_scores_z, 0.95))
threshold_raw = mu + threshold_z * std
print(f'Selected z-threshold: {threshold_z:.4f}  raw: {threshold_raw:.6f}')
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(tune_normal_scores_z, bins=50, alpha=0.55, label='Tune normal', color='steelblue', density=True)
if tune_defect_scores_z is not None:
    ax.hist(tune_defect_scores_z, bins=50, alpha=0.45, label='Tune defect', color='tomato', density=True)
ax.axvline(threshold_z, color='red', linewidth=2, linestyle='--', label=f'Threshold z={threshold_z:.3f}')
ax.set_xlabel('Anomaly score (z)')
ax.set_ylabel('Density')
ax.set_title('Score Distribution - Tune Split')
ax.legend()
plt.tight_layout()
fig.savefig(THRESHOLD_SELECTION_PNG_PATH, dpi=200, bbox_inches='tight')
plt.show()
plt.close(fig)
print(f'Saved threshold selection plot: {THRESHOLD_SELECTION_PNG_PATH}')


## Evaluate

This cell computes the final evaluation metrics on the held-out split using the selected decision threshold.


In [ ]:
from sklearn.metrics import classification_report
threshold_value = float(globals().get('threshold_z', globals().get('best_thresh')))
threshold_raw_value = float(globals().get('threshold_raw', threshold_value))
if 'test_normal_scores_z' in globals():
    normal_scores = np.asarray(test_normal_scores_z)
    defect_scores = np.asarray(test_defect_scores_z)
    score_column = 'score_z'
else:
    normal_scores = np.asarray(test_normal_scores)
    defect_scores = np.asarray(test_defect_scores)
    score_column = 'score'
TEST_SCORES_CSV_PATH = str(globals().get('TEST_SCORES_CSV_PATH', Path(RESULTS_DIR) / 'test_scores.csv'))
DEFECT_RECALL_CSV_PATH = str(globals().get('DEFECT_RECALL_CSV_PATH', Path(RESULTS_DIR) / 'test_defect_recall.csv'))
CLASSIFICATION_REPORT_PATH = str(globals().get('CLASSIFICATION_REPORT_PATH', Path(RESULTS_DIR) / 'classification_report.txt'))
CONFUSION_MATRIX_PNG_PATH = str(globals().get('CONFUSION_MATRIX_PNG_PATH', Path(PLOTS_DIR) / 'confusion_matrix_test.png'))
SCORE_DISTRIBUTION_PNG_PATH = str(globals().get('SCORE_DISTRIBUTION_PNG_PATH', Path(PLOTS_DIR) / 'score_distribution_test.png'))
y_true = np.concatenate([np.zeros(len(normal_scores), dtype=int), np.ones(len(defect_scores), dtype=int)])
scores = np.concatenate([normal_scores, defect_scores])
y_pred = (scores > threshold_value).astype(int)
roc_auc = float(roc_auc_score(y_true, scores))
report = classification_report(y_true, y_pred, target_names=['normal', 'anomaly'])
print(f'ROC-AUC : {roc_auc:.4f}')
print(f'Threshold: {threshold_value:.4f}   raw: {threshold_raw_value:.6f}')
print(report)
Path(CLASSIFICATION_REPORT_PATH).write_text(report, encoding='utf-8')
cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(4.8, 4.2))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, ax=ax, xticklabels=['pred normal', 'pred anomaly'], yticklabels=['true normal', 'true anomaly'])
ax.set_title('Confusion Matrix (Test)')
fig.tight_layout()
fig.savefig(CONFUSION_MATRIX_PNG_PATH, dpi=200, bbox_inches='tight')
plt.show()
plt.close(fig)
fig, ax = plt.subplots(figsize=(8, 4.5))
sns.kdeplot(normal_scores, label='Normal', fill=True, alpha=0.35, ax=ax)
sns.kdeplot(defect_scores, label='Defect', fill=True, alpha=0.35, ax=ax)
ax.axvline(threshold_value, color='red', linestyle='--', label=f'threshold={threshold_value:.2f}')
ax.set_xlabel('Anomaly score')
ax.set_ylabel('Density')
ax.set_title('Score Distribution (Test)')
ax.legend()
fig.tight_layout()
fig.savefig(SCORE_DISTRIBUTION_PNG_PATH, dpi=200, bbox_inches='tight')
plt.show()
plt.close(fig)
test_scores_df = pd.concat([test_normal_df[['failure_label', 'is_anomaly']].assign(split='test_normal'), test_defect_df[['failure_label', 'is_anomaly']].assign(split='test_defect')], ignore_index=True)
test_scores_df[score_column] = scores
test_scores_df['predicted_anomaly'] = y_pred
test_scores_df.to_csv(TEST_SCORES_CSV_PATH, index=False)
defect_recall_df = test_scores_df.loc[test_scores_df['is_anomaly'] == 1].groupby('failure_label').agg(count=('predicted_anomaly', 'count'), detected=('predicted_anomaly', 'sum'), recall=('predicted_anomaly', 'mean'), mean_score=(score_column, 'mean')).reset_index().sort_values(['recall', 'count', 'failure_label'], ascending=[True, False, True])
defect_recall_df.to_csv(DEFECT_RECALL_CSV_PATH, index=False)
print(defect_recall_df.round(3).to_string(index=False))


## UMAP Visualization

This cell projects saved embeddings for qualitative visualization and saves the resulting UMAP outputs.


In [ ]:
import subprocess
from sklearn.decomposition import PCA
UMAP_PNG_PATH = str(globals().get('UMAP_PNG_PATH', Path(PLOTS_DIR) / 'umap_test_embeddings.png'))
UMAP_CSV_PATH = str(globals().get('UMAP_CSV_PATH', Path(RESULTS_DIR) / 'umap_test_embeddings.csv'))
seed_value = int(globals().get('SEED', 42))
if 'test_normal_scores_z' in globals():
    normal_scores = np.asarray(test_normal_scores_z)
    defect_scores = np.asarray(test_defect_scores_z)
    threshold_value = float(globals().get('threshold_z', globals().get('best_thresh')))
    score_column = 'score_z'
else:
    normal_scores = np.asarray(test_normal_scores)
    defect_scores = np.asarray(test_defect_scores)
    threshold_value = float(globals().get('best_thresh', globals().get('threshold_z')))
    score_column = 'score'
if 'test_normal_loader' not in globals() or 'test_defect_loader' not in globals():
    batch_size = int(globals().get('BATCH_SIZE', globals().get('BANK_BATCH_SIZE', 128)))
    num_workers = int(globals().get('NUM_WORKERS', globals().get('BANK_WORKERS', 0)))
    _loader_kw = dict(batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=USE_CUDA, persistent_workers=num_workers > 0)
    test_normal_loader = DataLoader(WaferDataset(test_normal_df, IMAGE_SIZE), **_loader_kw)
    test_defect_loader = DataLoader(WaferDataset(test_defect_df, IMAGE_SIZE), **_loader_kw)
try:
    import umap.umap_ as umap
except Exception:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'umap-learn', '-q'])
    import umap.umap_ as umap

def collect_image_embeddings(loader, frame, split_name, desc):
    embeddings = []
    metadata_rows = []
    seen = 0
    with torch.inference_mode():
        for xb, yb in tqdm(loader, total=len(loader), desc=desc, unit='batch'):
            batch_size = len(yb)
            xb = xb.to(DEVICE)
            with torch.autocast('cuda', torch.float16, enabled=USE_CUDA):
                feat = extractor(xb)
                emb = extractor.proj(feat)
            img_emb = F.normalize(emb.float().mean(dim=1), p=2, dim=1)
            embeddings.append(img_emb.cpu().numpy())
            batch_meta = frame.iloc[seen:seen + batch_size][['failure_label', 'is_anomaly']].copy()
            batch_meta = batch_meta.reset_index(drop=True)
            batch_meta['split'] = split_name
            metadata_rows.append(batch_meta)
            seen += batch_size
    return (np.concatenate(embeddings, axis=0), pd.concat(metadata_rows, ignore_index=True))
X_normal, meta_normal = collect_image_embeddings(test_normal_loader, test_normal_df, 'test_normal', 'Embed test-normal')
X_defect, meta_defect = collect_image_embeddings(test_defect_loader, test_defect_df, 'test_defect', 'Embed test-defect')
X = np.concatenate([X_normal, X_defect], axis=0)
umap_df = pd.concat([meta_normal, meta_defect], ignore_index=True)
umap_df.insert(0, 'point_index', np.arange(len(umap_df)))
umap_df[score_column] = np.concatenate([normal_scores, defect_scores])
umap_df['predicted_anomaly'] = (umap_df[score_column] > threshold_value).astype(int)
umap_df['label_name'] = umap_df['is_anomaly'].map({0: 'normal', 1: 'defect'})
n_pca = min(32, X.shape[1], X.shape[0] - 1)
Xp = PCA(n_components=n_pca, random_state=seed_value).fit_transform(X)
reducer = umap.UMAP(n_components=2, n_neighbors=15, min_dist=0.1, metric='cosine', random_state=seed_value, transform_seed=seed_value, low_memory=True)
coords = reducer.fit_transform(Xp)
umap_df['umap_1'] = coords[:, 0]
umap_df['umap_2'] = coords[:, 1]
fig, ax = plt.subplots(figsize=(8, 6))
normal_mask = umap_df['is_anomaly'].to_numpy() == 0
defect_mask = ~normal_mask
ax.scatter(coords[normal_mask, 0], coords[normal_mask, 1], s=8, alpha=0.45, label='Normal', c='steelblue', linewidths=0)
ax.scatter(coords[defect_mask, 0], coords[defect_mask, 1], s=8, alpha=0.6, label='Defect', c='tomato', linewidths=0)
ax.set_title('UMAP of Test Image Embeddings')
ax.set_xlabel('UMAP-1')
ax.set_ylabel('UMAP-2')
ax.legend()
fig.tight_layout()
fig.savefig(UMAP_PNG_PATH, dpi=160, bbox_inches='tight')
plt.show()
plt.close(fig)
umap_df.to_csv(UMAP_CSV_PATH, index=False)
print(f'Saved UMAP figure: {UMAP_PNG_PATH}')
print(f'Saved UMAP points: {UMAP_CSV_PATH}')


## Save Outputs

This cell exports the trained model artifact, metrics, and any final bookkeeping files for reproducibility.


In [ ]:
import shutil
checkpoints_dir = Path(globals().get('CHECKPOINTS_DIR', globals().get('CHKPT_DIR', Path(ARTIFACT_DIR) / 'checkpoints')))
plots_dir = Path(globals().get('PLOTS_DIR', Path(ARTIFACT_DIR) / 'plots'))
results_dir = Path(globals().get('RESULTS_DIR', Path(ARTIFACT_DIR) / 'results'))
checkpoints_dir.mkdir(parents=True, exist_ok=True)
plots_dir.mkdir(parents=True, exist_ok=True)
results_dir.mkdir(parents=True, exist_ok=True)
MODEL_EXPORT_PATH = str(globals().get('MODEL_EXPORT_PATH', checkpoints_dir / 'model.pt'))
METRICS_EXPORT_PATH = str(globals().get('METRICS_EXPORT_PATH', results_dir / 'evaluation_metrics.json'))
TEST_SCORES_CSV_PATH = str(globals().get('TEST_SCORES_CSV_PATH', results_dir / 'test_scores.csv'))
DEFECT_RECALL_CSV_PATH = str(globals().get('DEFECT_RECALL_CSV_PATH', results_dir / 'test_defect_recall.csv'))
UMAP_PNG_PATH = str(globals().get('UMAP_PNG_PATH', plots_dir / 'umap_test_embeddings.png'))
UMAP_CSV_PATH = str(globals().get('UMAP_CSV_PATH', results_dir / 'umap_test_embeddings.csv'))
threshold_value = float(globals().get('threshold_z', globals().get('best_thresh', 0.0)))
threshold_raw_value = float(globals().get('threshold_raw', threshold_value))
artifact = {'threshold_z': threshold_value, 'threshold_raw': threshold_raw_value, 'config': {}}
for key in ['IMAGE_SIZE', 'VIT_FEATURE_BLOCK', 'VIT_FEATURE_BLOCKS', 'PATCH_EMBED_DIM', 'TOPK_PATCH_RATIO', 'PATCHCORE_NN_K']:
    if key in globals():
        artifact['config'][key.lower()] = globals()[key]
if 'mu' in globals():
    artifact['train_score_mu'] = float(mu)
if 'std' in globals():
    artifact['train_score_std'] = float(std)
if 'extractor' in globals():
    artifact['extractor_state_dict'] = extractor.state_dict()
elif 'model' in globals() and hasattr(model, 'state_dict'):
    artifact['model_state_dict'] = model.state_dict()
if 'memory_bank' in globals() and memory_bank is not None:
    artifact['memory_bank'] = memory_bank.cpu()
elif Path(MODEL_EXPORT_PATH).exists():
    try:
        existing_artifact = torch.load(MODEL_EXPORT_PATH, map_location='cpu')
        if isinstance(existing_artifact, dict):
            for key in ['memory_bank', 'extractor_state_dict', 'model_state_dict']:
                if key in existing_artifact and key not in artifact:
                    artifact[key] = existing_artifact[key]
    except Exception:
        pass
if artifact:
    torch.save(artifact, MODEL_EXPORT_PATH)
if 'test_scores_df' in globals():
    test_scores_df.to_csv(TEST_SCORES_CSV_PATH, index=False)
if 'defect_recall_df' in globals():
    defect_recall_df.to_csv(DEFECT_RECALL_CSV_PATH, index=False)
elif 'defect_df_out' in globals():
    defect_df_out.rename(columns={'failure_label': 'failure_label'}).to_csv(DEFECT_RECALL_CSV_PATH, index=False)
elif 'breakdown' in globals():
    breakdown.to_csv(DEFECT_RECALL_CSV_PATH, index=False)
metrics_payload = {'threshold_z': threshold_value, 'threshold_raw': threshold_raw_value, 'model_export_path': MODEL_EXPORT_PATH, 'test_scores_csv_path': TEST_SCORES_CSV_PATH, 'defect_recall_csv_path': DEFECT_RECALL_CSV_PATH, 'umap_png_path': UMAP_PNG_PATH, 'umap_csv_path': UMAP_CSV_PATH}
if 'roc_auc' in globals():
    metrics_payload['roc_auc_z'] = float(roc_auc)
if 'auroc' in globals():
    metrics_payload['roc_auc_z'] = float(auroc)
if 'cm' in globals():
    metrics_payload['confusion_matrix'] = cm.tolist()
Path(METRICS_EXPORT_PATH).write_text(json.dumps(metrics_payload, indent=2), encoding='utf-8')
print(f'Saved model artifact: {MODEL_EXPORT_PATH}')
print(f'Saved metrics: {METRICS_EXPORT_PATH}')


## Cleanup


In [ ]:
for name in ['memory_bank', 'memory_bank_t', 'train_scores', 'tune_normal_scores', 'tune_defect_scores', 'test_normal_scores', 'test_defect_scores', 'train_scores_z', 'tune_normal_scores_z', 'tune_defect_scores_z', 'test_normal_scores_z', 'test_defect_scores_z', 'scores', 'y_true', 'y_pred', 'train_loader', 'tune_normal_loader', 'tune_defect_loader', 'test_normal_loader', 'test_defect_loader']:
    if name in globals():
        del globals()[name]
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()
print('Cleared.')
